# LQR for the Cart-Pendulum System

<b> State Space Model </b> </br>

General Form: 

$ \dot{x} = Ax + Bu \\ 
y = Cx + Du $

We introduce 4 state variables $[x_{1}, x_{2}, x_{3}, x_{4}] = [x, \dot{x}, \theta, \dot{\theta}]$  </br>

$\dot{x}_{1} = \dot{x} = x_{2} \\ 
\dot{x}_{2} = \ddot{x} = \dfrac{u}{M} - \dfrac{mg}{M}x_{3} \\
\dot{x}_{3} = \dot{\theta} = x_{4} \\
\dot{x}_{4} = \ddot{\theta} = -\dfrac{u}{Ml} + \dfrac{(M+m)g}{Ml}x_{3} \\
y = [x_{1}, x_{2}, x_{3}, x_{4}]^{T}$

So, in vector-matrix form we have

$ \left[ \begin{array}{c} \dot{x}_1 \\ \dot{x}_2 \\ \dot{x}_3 \\ \dot{x}_4 \end{array} \right] = 
\begin{bmatrix} 0 & 1 & 0 & 0 \\ 
                0 & 0 & -\dfrac{mg}{M} & 0 \\
                0 & 0 & 0 & 1\\
                0 & 0 & \dfrac{(M+m)g}{Ml} & 0 \end{bmatrix} \left[ \begin{array}{c} x_1 \\ x_2 \\ x_3 \\ x_4 \end{array} \right] + \left[ \begin{array}{c} 0 \\ 1/M \\ 0 \\ -1/Ml \end{array} \right] u$ 


$y = \left[ \begin{array}{c} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{array} \right] \left[ \begin{array}{c} x_{1} \\ x_{2} \\x_{3} \\ x_{4} \end{array} \right] + \left[ \begin{array}{c} 0 \\ 0 \\ 0 \\ 0 \end{array} \right] u $

---

In [ ]:
%pip install control

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctrl
import scipy.linalg
import scipy.signal as signal
from control.matlab import *  # MATLAB-like functions

In [3]:
M = 2.0  # mass of the cart
m = 0.1   # ball mass
l = 0.25   # length of the rod
g = 9.8   # gravity

In [ ]:
A = np.array([[0, 1, 0, 0], 
              [0, 0, -3*m*g/(4*M+m), 0],
              [0, 0, 0, 1],
              [0, 0, 3*g*(M+m)/(l*(4*M+m)), 0]])

B = np.array([[0], [4/(4*M+m)], [0], [-3/(l*(4*M+m))]])

# C = np.array([[1, 0, 0, 0],
#               [0, 0, 1, 0],
#               [0, 0, 0, 1],
#               [0, 0, 0, 1]])
C = np.eye(4)

#D = np.array([[0], [0], [0], [0]])
D = np.zeros((4, 1))

print(A.shape, B.shape, C.shape, D.shape)
ssm_ct = ctrl.matlab.ss(A, B, C, D)
print(ssm_ct)

In [ ]:
# Controllability Matrix
Co = ctrl.ctrb(A, B)
Co_rank = np.linalg.matrix_rank(Co)
print("\nControllability Matrix:\n", Co)
print("\nRank of Controllability Matrix:", Co_rank)

# Observability Matrix
Ob = ctrl.obsv(A, C)
Ob_rank = np.linalg.matrix_rank(Ob)
print("\nObservability Matrix:\n", Ob)
print("\nRank of Observability Matrix:", Ob_rank)

In [ ]:
print(A)
print(B)
print(C)
print(D)

In [ ]:
# Define the Cost Matrices Q, R for the LQR
q1, q2, q3, q4 = 100, 0, 10, 0
r = 0.1
Q = np.diag([q1, q2, q3, q4])
R = np.array([[r]])
print(Q, Q.shape)
print(R, R.shape)

# Solve the Algebraic Riccati Equation (ARE)
K, P, E = ctrl.matlab.lqr(A, B, Q, R)
#K, P, E = ctrl.lqr(ssm_ct, Q, R)

print('\nGain Matrix:', K)
print('\nARE Solution:', P)
print('\nEigenvalues:', E)

In [ ]:
# Closed-loop system dynamics: x_dot = (A - B*K)x
Acl = A - B @ K
print("Acl = \n", Acl)
print("\nClosed-Loop eigenvalues:", np.linalg.eig(Acl)[0])
print("\nOpen-Loop eigenvalues:", np.linalg.eig(A)[0])

# plot the poles of the system
plt.figure(figsize=(5, 5))
plt.plot(np.real(np.linalg.eig(A)[0]), np.imag(np.linalg.eig(A)[0]), 'rx', label='Open-Loop')
plt.plot(np.real(np.linalg.eig(Acl)[0]), np.imag(np.linalg.eig(Acl)[0]), 'bo', markersize=4, label='Closed-Loop')
plt.axvline(x=0, color='k', linestyle='--')
plt.title("Poles of the system")
plt.xlabel("Re")
plt.ylabel("Im")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

sys_cl = ctrl.ss(Acl, B, C, D)
print("\n", sys_cl)

In [ ]:
# Simulation of the closed-loop system
time_signal = np.arange(0, 2, 0.01)
x_init = [0.0, 0.0, 0.0, 0.0]

# For a regulation problem, assume zero external input
#cl_input = np.array(600*[[0.0], [0.0], [0.0], [0.0]])
#u_zero = np.zeros_like(time_signal)

# STEP RESPONSE
u_step = -30*np.ones_like(time_signal)

yout, T, xout = ctrl.matlab.lsim(sys_cl, U=u_step, T=time_signal, X0=x_init)

# Compute control input at each time step: u = -K*x
u_control = - (K @ xout.T).flatten()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(T, yout[:, 0], 'b', label = 'x-LQR') # state1
plt.plot(T, yout[:, 1], 'r', label = 'dx-LQR') # state2
plt.xlabel("t")
plt.ylabel("x(t), dx(t)")
plt.grid()
plt.legend()
plt.tight_layout()
#plt.axis([0, 30, -0.5, 3])
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(T, yout[:, 2], 'g', label = 'θ-LQR') # state3
plt.plot(T, yout[:, 3], 'orange', label = 'dθ-LQR') # state4
plt.xlabel("t")
plt.ylabel("θ(t), dθ(t)")
plt.grid()
plt.legend()
plt.tight_layout()
#plt.axis([0, 30, -0.5, 3])
plt.show()

# # control input
plt.figure(figsize=(6, 4))
plt.plot(T, u_control, 'k', label = 'u-LQR')
plt.xlabel("t")
plt.ylabel("u(t)")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

---